In [1]:
import numpy as np
import re
import warnings
import time
import sys
from pathlib import Path

from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, average_precision_score)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

In [2]:
uiPrmdDir  = Path(r'C:\Users\sunil\OneDrive\Desktop\UI-PRMD')
kimoreDir  = Path(r'C:\Users\sunil\OneDrive\Desktop\KIMORE')
outDir     = Path(r'C:\Users\sunil\OneDrive\Desktop\iitbhu\processed_data')
outDir.mkdir(parents=True, exist_ok=True)

targetLen    = 150
butterCutoff = 6.0
butterFs     = 30.0
butterOrder  = 4
seed         = 42
np.random.seed(seed)

uiPrmdLowerJointIdx = [1, 2, 3, 5, 6, 7]
kimoreLowerJointIdx = [12, 13, 14, 16, 17, 18]
commonAngleTriplets = [(0, 1, 2), (3, 4, 5)]

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")
    sys.stdout.flush()

## Preprocessing & Feature Extraction

In [3]:
def resample_sequence(seq, targetLen):
    seq = np.asarray(seq, dtype=float)
    T, D = seq.shape
    if T == targetLen:
        return seq
    if T < 2:
        return np.tile(seq, (targetLen, 1))[:targetLen]
    xOld = np.linspace(0, 1, T)
    xNew = np.linspace(0, 1, targetLen)
    out = np.zeros((targetLen, D), dtype=float)
    for d in range(D):
        out[:, d] = interp1d(xOld, seq[:, d], kind='linear',
                             bounds_error=False,
                             fill_value=(seq[0, d], seq[-1, d]))(xNew)
    return out


def lowpass_filter(seq, cutoff=butterCutoff, fs=butterFs, order=butterOrder):
    nyq = 0.5 * fs
    b, a = butter(order, min(cutoff / nyq, 0.99), btype='low', analog=False)
    out = np.zeros_like(seq, dtype=float)
    for d in range(seq.shape[1]):
        out[:, d] = filtfilt(b, a, seq[:, d])
    return out


def remove_outliers(seq, zThresh=4.0):
    seq = np.nan_to_num(np.asarray(seq, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(seq, [5, 95], axis=0)
    mask = (seq >= lo) & (seq <= hi)
    clean = np.where(mask, seq, np.nan)
    mu = np.nanmean(clean, axis=0)
    sd = np.nanstd(clean, axis=0) + 1e-8
    return np.clip(seq, mu - zThresh * sd, mu + zThresh * sd)


def preprocess_sequence(seq, tLen=targetLen, applyFilter=True):
    seq = remove_outliers(seq)
    if applyFilter and seq.shape[0] > butterOrder * 3:
        seq = lowpass_filter(seq)
    return resample_sequence(seq, tLen)


def extract_lower_limb_joints(raw, jointIndices, coords=3):
    T = raw.shape[0]
    totalJoints = raw.shape[1] // coords
    bad = [i for i in jointIndices if i >= totalJoints]
    if bad:
        raise ValueError(f'{bad} out of range ({totalJoints} joints)')
    return raw.reshape(T, totalJoints, coords)[:, jointIndices, :].reshape(T, -1)


def per_subject_normalise(seq):
    mu = seq.mean(axis=0)
    sd = seq.std(axis=0) + 1e-8
    return (seq - mu) / sd


def extract_angle_series(joints, triplets):
    if not triplets:
        return np.zeros((joints.shape[0], 0))
    out = []
    for a, b, c in triplets:
        ba = joints[:, a, :] - joints[:, b, :]
        bc = joints[:, c, :] - joints[:, b, :]
        denom = (np.linalg.norm(ba, axis=1) * np.linalg.norm(bc, axis=1)) + 1e-8
        out.append(np.degrees(np.arccos(np.clip((ba * bc).sum(axis=1) / denom, -1.0, 1.0))))
    return np.vstack(out).T


def temporal_stats(s):
    return np.array([s.mean(), s.std(), s.min(), s.max(), s.max() - s.min(), float(np.median(s))])


def frequency_features(s, fs=butterFs):
    fftMag = np.abs(np.fft.rfft(s - s.mean()))
    freqs = np.fft.rfftfreq(len(s), d=1.0 / fs)
    domFreq = freqs[np.argmax(fftMag)] if len(fftMag) > 1 else 0.0
    bandPower = float(fftMag[(freqs >= 0) & (freqs <= 3.0)].sum())
    return np.array([domFreq, bandPower])


def zero_crossing_rate(s):
    centered = s - s.mean()
    return float((np.diff(np.sign(centered)) != 0).sum() / max(len(s), 1))


def extract_features(seq, nJoints, coords=3, angleTriplets=None):
    T = seq.shape[0]
    joints = seq.reshape(T, nJoints, coords)
    feats, names = [], []
    axesLbl = ['x', 'y', 'z'][:coords]

    for stat, sname in [
        (joints.mean(axis=0).flatten(), 'mean'),
        (joints.std(axis=0).flatten(), 'std'),
        ((joints.max(axis=0) - joints.min(axis=0)).flatten(), 'range'),
        (np.abs(np.diff(joints, n=2, axis=0)).mean(axis=0).flatten(), 'smooth'),
    ]:
        feats.append(stat)
        names += [f'j{j}_{ax}_{sname}' for j in range(nJoints) for ax in axesLbl]

    pairCount = nJoints // 2
    if pairCount:
        sym = np.abs(joints[:, 0:2 * pairCount:2, :] - joints[:, 1:2 * pairCount:2, :]).mean(axis=0).flatten()
        feats.append(sym)
        names += [f'sym_{i}' for i in range(len(sym))]

    if angleTriplets:
        valid = [t for t in angleTriplets if max(t) < nJoints]
        angS = extract_angle_series(joints, valid)
        for k, (a, b, c) in enumerate(valid):
            s = angS[:, k]
            feats.append(np.concatenate([temporal_stats(s), frequency_features(s), [zero_crossing_rate(s)]]))
            pfx = f'angle_{a}_{b}_{c}'
            names += [f'{pfx}_{n}' for n in ['mean', 'std', 'min', 'max', 'range', 'median', 'domFreq', 'bandPower', 'zcr']]

    for arr, tag in [(np.diff(seq, n=1, axis=0), 'vel'), (np.diff(seq, n=2, axis=0), 'acc')]:
        mag = np.linalg.norm(arr, axis=1)
        feats.append(temporal_stats(mag))
        names += [f'{tag}_mag_{n}' for n in ['mean', 'std', 'min', 'max', 'range', 'median']]

    return np.concatenate(feats), names

## Data Augmentation

In [4]:
def aug_gaussian_noise(seq, sigma=0.01):
    return seq + np.random.normal(0, sigma, seq.shape)


def aug_time_warp(seq, sigma=0.2):
    T, D = seq.shape
    knots = np.linspace(0, 1, 6)
    warpPts = knots + np.random.normal(0, sigma, len(knots))
    warpPts[0] = 0.0
    warpPts[-1] = 1.0
    warpPts = np.clip(np.sort(warpPts), 0, 1)
    xOld = np.linspace(0, 1, T)
    xWarp = interp1d(knots, warpPts, kind='cubic')(xOld)
    xWarp = np.clip(xWarp, 0, 1)
    out = np.zeros_like(seq)
    for d in range(D):
        out[:, d] = interp1d(xOld, seq[:, d], bounds_error=False,
                             fill_value=(seq[0, d], seq[-1, d]))(xWarp)
    return out


def aug_mirror(seq, nJoints, coords=3):
    T = seq.shape[0]
    joints = seq.reshape(T, nJoints, coords).copy()
    pairCount = nJoints // 2
    for i in range(pairCount):
        joints[:, i, :], joints[:, pairCount + i, :] = (
            joints[:, pairCount + i, :].copy(), joints[:, i, :].copy())
    joints[:, :, 0] *= -1
    return joints.reshape(T, -1)


def aug_speed(seq, factor=None):
    if factor is None:
        factor = np.random.choice([0.8, 1.2])
    T, D = seq.shape
    newT = max(2, int(T * factor))
    xOld = np.linspace(0, 1, T)
    xNew = np.linspace(0, 1, newT)
    stretched = np.zeros((newT, D))
    for d in range(D):
        stretched[:, d] = interp1d(xOld, seq[:, d],
                                   bounds_error=False,
                                   fill_value=(seq[0, d], seq[-1, d]))(xNew)
    return resample_sequence(stretched, T)


def augment_dataset(xSeq, y, nJoints, coords=3):
    augX, augY = [], []
    rng = np.random.default_rng(seed)
    for seq, label in zip(xSeq, y):
        for fn in [
            lambda s: aug_gaussian_noise(s, sigma=0.01),
            lambda s: aug_time_warp(s, sigma=0.15),
            lambda s: aug_mirror(s, nJoints=nJoints, coords=coords),
            lambda s: aug_speed(s, factor=float(rng.choice([0.8, 1.2]))),
        ]:
            try:
                augX.append(fn(seq))
                augY.append(label)
            except Exception:
                pass
    return np.array(augX), np.array(augY)

## Dataset Loaders

In [5]:
def read_skeleton_file(path, coords=3):
    lines = [l.strip() for l in Path(path).read_text().splitlines() if l.strip()]
    idx = 0
    nFrames = int(float(lines[idx])); idx += 1
    frames = []
    for _ in range(nFrames):
        if idx >= len(lines):
            break
        nBodies = int(float(lines[idx])); idx += 1
        if nBodies <= 0:
            continue
        idx += 1
        nJoints = int(float(lines[idx])); idx += 1
        joints = []
        for _j in range(nJoints):
            vals = [float(x) for x in lines[idx].split()]
            idx += 1
            joints.extend(vals[:coords])
        for _b in range(1, nBodies):
            idx += 1
            extraNj = int(float(lines[idx])); idx += 1
            idx += extraNj
        frames.append(joints)
    if not frames:
        raise ValueError(f'no frames: {path}')
    return np.asarray(frames, dtype=float)


def parse_uiprmd_name(path):
    stem = path.stem
    m = re.match(r'[Aa](\d+)[Ss](\d+)[Ee](\d+)[Cc](\d+)', stem)
    if m:
        action, subject, episode, correctness = map(int, m.groups())
        return {'action': action, 'subject': subject, 'episode': episode,
                'label': 1 if correctness == 1 else 0}
    m2 = re.match(r'[Mm]_(\d+)_(\d+)', stem)
    if m2:
        action, subject = map(int, m2.groups())
        parts = [p.lower() for p in path.parts]
        label = 1 if any('correct' in p and 'incorrect' not in p for p in parts) else 0
        return {'action': action, 'subject': subject, 'episode': 1, 'label': label}
    parts = [p.lower() for p in path.parts]
    label = 1 if any('correct' in p and 'incorrect' not in p for p in parts) else 0
    subject = next((int(p) for p in reversed(path.parts[:-1]) if p.isdigit()), 0)
    return {'action': 0, 'subject': subject, 'episode': 1, 'label': label}


def parse_kimore_name(path):
    stem = path.stem
    m = re.match(r'[Gg](\d+)[Ss](\d+)[Ee](\d+)[Rr](\d+)', stem)
    if m:
        g, s, e, r = map(int, m.groups())
        return {'group_code': g, 'subject': g * 1000 + s, 'subject_in_group': s,
                'exercise': e, 'repetition': r, 'label': e}
    parts = path.parts
    group, subjectNum, exercise, repetition = 0, 0, 0, 1
    for part in parts:
        mg = re.match(r'[Gg]roup(\d+)', part, re.I)
        if mg:
            group = int(mg.group(1))
        ms = re.match(r'[Ss]ubj(?:ect)?(\d+)', part, re.I)
        if ms:
            subjectNum = int(ms.group(1))
        me = re.match(r'[Ee]s?(\d+)', part, re.I)
        if me:
            exercise = int(me.group(1))
        mr = re.match(r'[Rr]ep(?:etition)?(\d+)', part, re.I)
        if mr:
            repetition = int(mr.group(1))
    if subjectNum == 0:
        subjectNum = next((int(p) for p in reversed(parts[:-1]) if p.isdigit()), 0)
    return {'group_code': group, 'subject': group * 1000 + subjectNum,
            'subject_in_group': subjectNum, 'exercise': exercise,
            'repetition': repetition, 'label': exercise}


def _load_dataset(baseDir, files, parseNameFn, lowerJointIdx, datasetName, labelType):
    xFeats, xSeqs, yAll, groupsAll, infoRows = [], [], [], [], []
    featureNames = None
    expectedFlen = None
    skipped = 0
    nJointsLl = len(lowerJointIdx)
    total = len(files)

    for i, fpath in enumerate(files):
        if (i + 1) % 200 == 0 or i == 0:
            log(f"{datasetName}: {i + 1}/{total}")
        try:
            meta = parseNameFn(fpath)
            raw = read_skeleton_file(fpath)
            raw = raw[:, :(raw.shape[1] // 3) * 3]
            if raw.shape[1] == 0:
                raise ValueError('no coords')
            ll = extract_lower_limb_joints(raw, lowerJointIdx, coords=3)
            seq = preprocess_sequence(ll, applyFilter=True)
            seqNorm = per_subject_normalise(seq)
            fvec, fnames = extract_features(seqNorm, nJointsLl, 3, commonAngleTriplets)
            if expectedFlen is not None and len(fvec) != expectedFlen:
                raise ValueError(f'flen mismatch {len(fvec)}')
            expectedFlen = len(fvec)
            xFeats.append(fvec)
            xSeqs.append(seqNorm)
            yAll.append(meta['label'])
            groupsAll.append(meta['subject'])
            row = {'dataset': datasetName, 'subject': meta['subject'],
                   'label': meta['label'], 'source_file': str(fpath),
                   'n_raw_frames': raw.shape[0]}
            if 'action' in meta:
                row['exercise'] = f"A{meta['action']:02d}"
            if 'episode' in meta:
                row['episode'] = meta['episode']
            if 'exercise' in meta:
                row['exercise'] = f"E{meta['exercise']:03d}"
            if 'repetition' in meta:
                row['repetition'] = meta['repetition']
            if 'group_code' in meta:
                row['group_code'] = meta['group_code']
            row['label_meaning'] = ('correct' if meta['label'] == 1 else 'incorrect')                 if labelType == 'binary_quality' else f"exercise_{meta['label']:03d}"
            infoRows.append(row)
            if featureNames is None:
                featureNames = fnames
        except Exception:
            skipped += 1

    if not xFeats:
        log(f'{datasetName}: no samples')
        return None

    log(f"{datasetName}: {len(xFeats)} ok / {skipped} skipped / {len(set(groupsAll))} subj / {expectedFlen} feats")
    return {
        'X': np.array(xFeats),
        'X_seq': np.array(xSeqs),
        'y': np.array(yAll),
        'groups': np.array(groupsAll),
        'feature_names': featureNames,
        'meta': infoRows,
        'dataset': datasetName,
        'label_type': labelType,
        'n_joints': nJointsLl,
    }


def load_and_process_uiprmd(baseDir):
    baseDir = Path(baseDir)
    if not baseDir.exists():
        log(f'UI-PRMD missing: {baseDir}')
        return None
    files = sorted(baseDir.glob('**/*.skeleton'))
    if not files:
        log(f'UI-PRMD: no .skeleton files')
        return None
    log(f'UI-PRMD: {len(files)} files')
    return _load_dataset(baseDir, files, parse_uiprmd_name,
                         uiPrmdLowerJointIdx, 'UI-PRMD', 'binary_quality')


def load_and_process_kimore(baseDir):
    baseDir = Path(baseDir)
    if not baseDir.exists():
        log(f'KIMORE missing: {baseDir}')
        return None
    files = sorted(baseDir.glob('**/*.skeleton'))
    if not files:
        log(f'KIMORE: no .skeleton files')
        return None
    log(f'KIMORE: {len(files)} files')
    return _load_dataset(baseDir, files, parse_kimore_name,
                         kimoreLowerJointIdx, 'KIMORE', 'multiclass_exercise')

## MediaPipe Log → Sequence Bridge

In [6]:
def mediapipe_log_to_sequences(logLKnee, logRKnee, logLHip, logRHip,
                                subjectId='unknown', exerciseType='unknown',
                                window=60, stride=30):
    angles = np.vstack([logLKnee, logRKnee, logLHip, logRHip]).T
    T = angles.shape[0]
    if T < window:
        angles = np.pad(angles, ((0, window - T), (0, 0)), mode='edge')
        T = window

    sequences, features, metadata = [], [], []
    for start in range(0, T - window + 1, stride):
        end = start + window
        segment = angles[start:end]
        segClean = remove_outliers(segment)
        if segClean.shape[0] > butterOrder * 3:
            segClean = lowpass_filter(segClean)
        seqNorm = resample_sequence(segClean, targetLen)
        featsList, featNames = [], []
        angleNames = ['L_knee', 'R_knee', 'L_hip', 'R_hip']
        for i, name in enumerate(angleNames):
            s = seqNorm[:, i]
            featsList.append(temporal_stats(s))
            featNames += [f'{name}_{stat}' for stat in ['mean', 'std', 'min', 'max', 'range', 'median']]
            featsList.append(frequency_features(s))
            featNames += [f'{name}_domFreq', f'{name}_bandPower']
            featsList.append([zero_crossing_rate(s)])
            featNames += [f'{name}_zcr']
        fvec = np.concatenate(featsList)
        sequences.append(seqNorm)
        features.append(fvec)
        metadata.append({'subject': subjectId, 'exercise': exerciseType,
                         'start_frame': start, 'end_frame': end})

    return {'X_seq': np.array(sequences), 'X': np.array(features),
            'meta': metadata, 'feature_names': featNames}

## LOSO Cross-Validation & Per-Dataset Evaluation

In [7]:
def _plot_cm(cm, labels, title, savePath):
    fig, ax = plt.subplots(figsize=(max(4, len(labels)), max(3, len(labels))))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('predicted')
    ax.set_ylabel('true')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(savePath, dpi=120)
    plt.close(fig)


def evaluate_model_loso(X, y, groups, nJoints, xSeq=None, bestParams=None):
    yTrueAll, yPredAll = [], []
    logo = LeaveOneGroupOut()
    nFolds = logo.get_n_splits(groups=groups)
    log(f'LOSO {nFolds} folds')
    isBinary = len(np.unique(y)) == 2

    for fold, (trainIdx, testIdx) in enumerate(logo.split(X, y, groups)):
        if (fold + 1) % 10 == 0 or fold == 0:
            log(f'  fold {fold + 1}/{nFolds}')
        xTrain, xTest = X[trainIdx], X[testIdx]
        yTrain, yTest = y[trainIdx], y[testIdx]

        if xSeq is not None:
            xSeqTrain = xSeq[trainIdx]
            augXSeq, augY = augment_dataset(xSeqTrain, yTrain, nJoints)
            if len(augXSeq):
                augXFeat = np.array([
                    extract_features(augXSeq[k], nJoints, 3, commonAngleTriplets)[0]
                    for k in range(len(augXSeq))
                ])
                augXFeat = np.nan_to_num(augXFeat)
                if augXFeat.shape[1] == xTrain.shape[1]:
                    xTrain = np.vstack([xTrain, augXFeat])
                    yTrain = np.concatenate([yTrain, augY])

        params = bestParams if bestParams else {}
        clf = RandomForestClassifier(n_estimators=params.get('n_estimators', 100),
                                     max_depth=params.get('max_depth', None),
                                     min_samples_split=params.get('min_samples_split', 2),
                                     random_state=seed, class_weight='balanced', n_jobs=-1)
        clf.fit(xTrain, yTrain)
        preds = clf.predict(xTest)
        yTrueAll.extend(yTest)
        yPredAll.extend(preds)

    yTrue = np.array(yTrueAll)
    yPred = np.array(yPredAll)
    report = classification_report(yTrue, yPred, zero_division=0)
    cm = confusion_matrix(yTrue, yPred)
    labels = np.unique(yTrue).tolist()

    print(report)
    log(f'CM: {cm}')

    extraMetrics = {}
    if isBinary:
        extraMetrics['roc_auc'] = roc_auc_score(yTrue, yPred)
        extraMetrics['pr_auc'] = average_precision_score(yTrue, yPred)
        log(f"ROC-AUC={extraMetrics['roc_auc']:.4f}  PR-AUC={extraMetrics['pr_auc']:.4f}")
    else:
        extraMetrics['roc_auc_ovr'] = roc_auc_score(
            yTrue, np.eye(len(labels))[yPred], multi_class='ovr', average='macro')
        log(f"ROC-AUC(ovr)={extraMetrics['roc_auc_ovr']:.4f}")

    perSubjAcc = {}
    for subj in np.unique(groups):
        mask = groups == subj
        perSubjAcc[int(subj)] = float((yTrue[mask] == yPred[mask]).mean())
    log(f"per-subject acc  min={min(perSubjAcc.values()):.3f}  max={max(perSubjAcc.values()):.3f}")

    return report, cm, labels, extraMetrics, perSubjAcc


def evaluate_per_dataset(datasets, scaler, bestParams):
    perDsResults = {}
    for ds in datasets:
        name = ds['dataset']
        log(f'per-dataset LOSO: {name}')
        xDs = np.nan_to_num(scaler.transform(ds['X']), nan=0.0, posinf=0.0, neginf=0.0)
        report, cm, labels, extraMetrics, perSubjAcc = evaluate_model_loso(
            xDs, ds['y'], ds['groups'], ds['n_joints'],
            xSeq=ds['X_seq'], bestParams=bestParams)
        perDsResults[name] = {'report': report, 'cm': cm, 'labels': labels,
                              'extra': extraMetrics, 'per_subj': perSubjAcc}
    return perDsResults

## Cross-Dataset Generalisation

In [8]:
def cross_dataset_eval(datasets, scaler, bestParams):
    if len(datasets) < 2:
        log('need >=2 datasets for cross-dataset eval')
        return {}

    crossResults = {}
    for i, trainDs in enumerate(datasets):
        for j, testDs in enumerate(datasets):
            if i == j:
                continue
            trainName = trainDs['dataset']
            testName  = testDs['dataset']
            key = f'{trainName}→{testName}'
            log(f'cross-dataset: train={trainName} test={testName}')

            xTrain = np.nan_to_num(scaler.transform(trainDs['X']), nan=0.0, posinf=0.0, neginf=0.0)
            xTest  = np.nan_to_num(scaler.transform(testDs['X']),  nan=0.0, posinf=0.0, neginf=0.0)
            yTrain = trainDs['y']
            yTest  = testDs['y']

            sharedLabels = np.intersect1d(np.unique(yTrain), np.unique(yTest))
            if len(sharedLabels) == 0:
                log(f'  {key}: no shared labels, skipping')
                continue

            trainMask = np.isin(yTrain, sharedLabels)
            testMask  = np.isin(yTest,  sharedLabels)
            xTrain, yTrain = xTrain[trainMask], yTrain[trainMask]
            xTest,  yTest  = xTest[testMask],   yTest[testMask]

            params = bestParams if bestParams else {}
            clf = RandomForestClassifier(
                n_estimators=params.get('n_estimators', 100),
                max_depth=params.get('max_depth', None),
                min_samples_split=params.get('min_samples_split', 2),
                random_state=seed, class_weight='balanced', n_jobs=-1)
            clf.fit(xTrain, yTrain)
            yPred = clf.predict(xTest)

            report = classification_report(yTest, yPred, zero_division=0)
            cm = confusion_matrix(yTest, yPred)
            isBinary = len(sharedLabels) == 2

            print(f'--- {key}')
            print(report)

            extraMetrics = {}
            if isBinary:
                extraMetrics['roc_auc'] = roc_auc_score(yTest, yPred)
                extraMetrics['pr_auc'] = average_precision_score(yTest, yPred)
                log(f"  ROC-AUC={extraMetrics['roc_auc']:.4f}  PR-AUC={extraMetrics['pr_auc']:.4f}")

            crossResults[key] = {'report': report, 'cm': cm,
                                  'labels': sharedLabels.tolist(), 'extra': extraMetrics}
    return crossResults

## Hyperparameter Search

In [9]:
def run_hyperparameter_search(X, y, groups):
    log('RandomizedSearchCV')
    paramDist = {
        'n_estimators': [50, 100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 'log2', 0.3],
    }
    baseClf = RandomForestClassifier(class_weight='balanced', random_state=seed, n_jobs=-1)
    logo = LeaveOneGroupOut()
    nCv = min(logo.get_n_splits(groups=groups), 10)
    cvSplits = list(logo.split(X, y, groups))[:nCv]

    search = RandomizedSearchCV(
        baseClf, paramDist, n_iter=20, cv=cvSplits,
        scoring='f1_macro', random_state=seed, n_jobs=-1, verbose=0)
    search.fit(X, y)

    log(f'best f1_macro={search.best_score_:.4f}')
    log(f'best params={search.best_params_}')
    return search.best_params_

## Feature Importance

In [10]:
def compute_feature_importance(clf, featureNames, topN=20):
    importances = pd.Series(clf.feature_importances_, index=featureNames)
    topFeatures = importances.nlargest(topN)

    fig, ax = plt.subplots(figsize=(8, max(4, topN // 2)))
    topFeatures.sort_values().plot.barh(ax=ax)
    ax.set_title(f'top {topN} features')
    ax.set_xlabel('importance')
    fig.tight_layout()
    fig.savefig(outDir / 'feature_importance.png', dpi=120)
    plt.close(fig)

    topFeatures.to_csv(outDir / 'top_features.csv')
    log(f'top {topN} features saved')
    return importances

## Main Pipeline

In [ ]:
def align_features(datasets):
    maxLen = max(d['X'].shape[1] for d in datasets)
    aligned = []
    for d in datasets:
        xD = d['X']
        if xD.shape[1] < maxLen:
            xD = np.hstack([xD, np.zeros((xD.shape[0], maxLen - xD.shape[1]))])
        copied = dict(d)
        copied['X'] = xD
        aligned.append(copied)
    return aligned, maxLen


def main():
    tStart = time.time()
    log(outDir.resolve())

    uiPrmdData = load_and_process_uiprmd(uiPrmdDir)
    kimoreData = load_and_process_kimore(kimoreDir)

    allDatasets = [d for d in [uiPrmdData, kimoreData] if d is not None]
    if not allDatasets:
        log('no datasets')
        return

    for ds in allDatasets:
        dsName = ds['dataset'].lower().replace('-', '_')
        np.save(outDir / f'X_seq_{dsName}.npy', ds['X_seq'])
        log(f"{ds['dataset']} seq {ds['X_seq'].shape}")

    alignedDatasets, maxFeatLen = align_features(allDatasets)
    xAll      = np.vstack([d['X'] for d in alignedDatasets])
    yAll      = np.concatenate([d['y'] for d in alignedDatasets])
    groupsAll = np.concatenate([d['groups'] for d in alignedDatasets])
    metaAll   = [row for d in alignedDatasets for row in d['meta']]
    featureNamesAll = allDatasets[0]['feature_names']
    nJointsAll = allDatasets[0]['n_joints']

    log(f'X={xAll.shape}  y={yAll.shape}  subj={len(np.unique(groupsAll))}')
    log(f'labels={dict(zip(*np.unique(yAll, return_counts=True)))}')

    scaler = StandardScaler()
    xNorm = np.nan_to_num(scaler.fit_transform(xAll), nan=0.0, posinf=0.0, neginf=0.0)
    log(f'scaled {xNorm.shape}')

    bestParams = run_hyperparameter_search(xNorm, yAll, groupsAll)

    log('LOSO combined')
    xSeqAll = np.vstack([d['X_seq'] for d in alignedDatasets])
    report, cm, labels, extraMetrics, perSubjAcc = evaluate_model_loso(
        xNorm, yAll, groupsAll, nJointsAll, xSeq=xSeqAll, bestParams=bestParams)

    _plot_cm(cm, labels, 'LOSO combined', outDir / 'cm_combined.png')

    log('per-dataset LOSO')
    perDsResults = evaluate_per_dataset(alignedDatasets, scaler, bestParams)
    for dsName, res in perDsResults.items():
        _plot_cm(res['cm'], res['labels'], f'LOSO {dsName}', outDir / f'cm_{dsName.lower().replace("-","_")}.png')

    log('cross-dataset')
    crossResults = cross_dataset_eval(alignedDatasets, scaler, bestParams)
    for key, res in crossResults.items():
        safeName = key.replace('\u2192', '_to_').replace('-', '_').lower()
        _plot_cm(res['cm'], res['labels'], key, outDir / f'cm_{safeName}.png')

    log('final model')
    clfFinal = RandomForestClassifier(
        n_estimators=bestParams.get('n_estimators', 100),
        max_depth=bestParams.get('max_depth', None),
        min_samples_split=bestParams.get('min_samples_split', 2),
        random_state=seed, class_weight='balanced', n_jobs=-1)
    clfFinal.fit(xNorm, yAll)

    importances = compute_feature_importance(clfFinal, featureNamesAll)

    joblib.dump(clfFinal, outDir / 'baseline_random_forest.joblib')
    joblib.dump(scaler,   outDir / 'feature_scaler.joblib')

    np.save(outDir / 'X_features.npy', xNorm)
    np.save(outDir / 'y_labels.npy', yAll)
    np.save(outDir / 'groups.npy', groupsAll)

    with open(outDir / 'feature_names.json', 'w') as f:
        json.dump(featureNamesAll, f, indent=2)
    log(f'feature_names.json  {len(featureNamesAll)} cols')

    metaDf = pd.DataFrame(metaAll)
    metaDf.insert(0, 'row_index', range(len(metaDf)))
    metaDf.to_csv(outDir / 'meta.csv', index=False)

    with open(outDir / 'evaluation_report.txt', 'w') as f:
        f.write(report)
        f.write('\nextra metrics\n')
        for k, v in extraMetrics.items():
            f.write(f'{k}={v:.4f}\n')
        f.write('\ncross-dataset\n')
        for key, res in crossResults.items():
            f.write(f'\n{key}\n')
            f.write(res['report'])
            f.write('\n')

    scalerDatasetsStr = ', '.join(d['dataset'] for d in allDatasets)
    readmeLines = [
        'X_features.npy         float64 array (n_samples, n_features)  StandardScaler z-scored',
        'y_labels.npy           int array (n_samples,)',
        'groups.npy             int array (n_samples,)  subject IDs',
        f'feature_names.json     ordered list of {len(featureNamesAll)} column names matching X_features.npy',
        'meta.csv               one row per sample; row_index matches X_features.npy row exactly',
        f'feature_scaler.joblib  StandardScaler fit on combined data from: {scalerDatasetsStr}',
        '                       apply this scaler to any new data before inference',
        'baseline_random_forest.joblib  RandomForest trained on full combined scaled data',
        f'X_seq_*.npy            raw normalised sequences shape (n_samples, {targetLen}, n_joints*3)',
        'cm_*.png               confusion matrix plots',
        'feature_importance.png / top_features.csv  feature ranking from final model',
        'evaluation_report.txt  LOSO classification report + extra metrics + cross-dataset results',
    ]
    with open(outDir / 'README.txt', 'w') as f:
        f.write('\n'.join(readmeLines))
    log('README.txt')

    elapsed = time.time() - tStart
    log(f'{xNorm.shape[0]} samples  {xNorm.shape[1]} feats  {len(np.unique(groupsAll))} subj  {elapsed:.1f}s')


if __name__ == '__main__':
    main()

[11:49:13] C:\Users\sunil\OneDrive\Desktop\iitbhu\processed_data
[11:49:13] UI-PRMD: 1326 files
[11:49:13] UI-PRMD: 1/1326
[11:49:18] UI-PRMD: 200/1326
[11:49:23] UI-PRMD: 400/1326
[11:49:29] UI-PRMD: 600/1326
[11:49:34] UI-PRMD: 800/1326
[11:49:39] UI-PRMD: 1000/1326
[11:49:44] UI-PRMD: 1200/1326
[11:49:47] UI-PRMD: 1326 ok / 0 skipped / 10 subj / 111 feats
[11:49:47] KIMORE: 2806 files
[11:49:47] KIMORE: 1/2806
[11:49:53] KIMORE: 200/2806
[11:49:58] KIMORE: 400/2806


In [ ]:
main()